# Assignment 4. Pruning for LLM

> ### 이 노트북 읽는 법
> 이 파일은 `4. Pruning for LLM_answer.ipynb`에 **파이썬 초보자용 해설**을 덧붙인 버전입니다.
> - 원본의 셀(설명·코드·실행 결과)은 **그대로** 두었습니다.
> - 번호가 붙은 코드 셀 **바로 아래**마다 `📘 셀 N 해설` 칸을 추가했습니다. 맨 앞 **Setup 부분(셀 1 ~ 3)은 해설을 생략**했고, 본문인 셀 4 ~ 10을 설명합니다.
> - 해설은 **한 줄 요약 → 코드 뜯어보기 → 출력 해석** 순서입니다.
> - 원본 파일에는 **셀 6 ~ 10의 실행 결과가 저장되어 있지 않습니다**(실행하지 않은 상태로 저장됨). 해당 셀의 해설은 코드 설명 위주입니다.
>
> **Setup에서 만들어 두고 뒤에서 계속 쓰는 것들**
> - `model_path` : 사용할 모델 이름 (`HuggingFaceTB/SmolLM-135M`, 파라미터 약 1억 3,500만 개의 작은 언어 모델)
> - `tokenizer` : 문장을 모델이 이해하는 숫자 조각(**토큰**)으로 바꿔 주는 도구
> - `model` : 불러온 언어 모델 (가중치는 16비트 실수, FP16)
> - `evaluate(model, tokenizer)` : 모델의 **perplexity**를 계산하는 함수. 원본 모델의 값은 **16.54**입니다.
>
> **미리 알아 둘 용어**
> - **LLM(대규모 언어 모델)** : 앞 단어들을 보고 다음 단어를 예측하도록 학습된 모델
> - **Perplexity(혼란도)** : 언어 모델의 성능 지표. 다음 단어를 고를 때 "몇 개의 후보 사이에서 헷갈리는가"에 해당하며, **낮을수록 좋습니다.**
> - **프루닝(Pruning)** : 덜 중요한 가중치를 0으로 만들어 모델을 가볍게 하는 기법 / **Sparsity** : 0으로 만든 비율
> - **Linear 층** : `출력 = 입력 × 가중치 행렬`을 계산하는 층. LLM의 가중치 대부분이 여기에 있습니다.
> - **activation** : 층에 들어가거나 층에서 나오는 중간 계산 값

## Goals

본 실습에서는 대규모 언어 모델(Large Language Model, LLM)의 크기를 효과적으로 줄이는 Pruning 기법을 학습합니다. 특히 Magnitude-based pruning과 최근 주목받는 Wanda 기법을 활용하여, 파라미터의 수를 효율적으로 감소시키면서 모델의 성능을 유지하는 방법을 실습합니다.


## Contents

1. **Magnitude-based Pruning 실습**:
   - 간단한 magnitude 기반 pruning 방법을 통해 모델 크기를 감소시키고 성능 변화를 확인합니다.
2. **Wanda를 이용한 Pruning 실습**:
   - Activation의 중요도를 측정하여 더 정교하게 pruning을 수행하는 Wanda 방법을 구현하고 모델의 성능을 비교합니다.

## Setup

필요한 모듈을 불러옵니다.

In [1]:
# 1
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os

### 모델 평가

Wikitext-2 데이터셋을 사용하여 모델의 성능을 평가하는 지표인 perplexity를 계산합니다.

**Perplexity란?**
- Perplexity는 언어 모델이 주어진 텍스트를 얼마나 잘 예측하는지를 수치로 나타낸 지표입니다.
- 수학적으로는 모델이 예측한 확률분포의 "불확실성"을 측정하는 값이며, 값이 **낮을수록 모델의 성능이 좋다**고 해석합니다.
- 단어 $\{w_1, w_2, w_3, ..., w_N\}$으로 구성된 문장의 Perplexity는 다음과 같은 수식으로 나타낼 수 있습니다.
    - $Perplexity = \sqrt[N]{\frac{1}{\prod_{i=1}^{N} P(w_i | w_1, w_2, ..., w_{i-1})}}$
    - 여기서 $P(w_i | w_1, w_2, ..., w_{i-1})$은 $i$번째에 $w_i$ 단어를 생성할 확률을 의미합니다.
    - 즉, Perplexity는 문자의 발생 확률에 대한 역수를 의미하게 됩니다.
- 직관적으로 Perplexity가 10이라면 모델이 다음 단어 후보를 10개 정도로 생각하고 있다고 볼 수 있습니다.

In [2]:
# 2
def evaluate(model, tokenizer):
    """
    모델의 perplexity를 계산하는 함수입니다.

    Args:
        model: 평가할 모델
        tokenizer: 토크나이저

    Returns:
        float: 계산된 perplexity 값
    """
    # 테스트 데이터셋 로드 및 전처리
    testenc = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')

    # 입력 데이터를 모델 디바이스로 이동
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()  # 평가 모드로 설정

    # Negative log likelihood 계산
    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating..."):
        # 배치 데이터 준비
        batch = testenc[:, (i * 2048):((i + 1) * 2048)].to(model.device)

        # 모델 추론
        with torch.no_grad():
            lm_logits = model(batch).logits

        # 로짓과 레이블 시프트
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 2048):((i + 1) * 2048)][:, 1:]

        # 손실 계산
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 2048
        nlls.append(neg_log_likelihood)

    # Perplexity 계산 및 반환
    return torch.exp(torch.stack(nlls).sum() / (nsamples * 2048))

### SmolLM-135M 모델 로딩

SmolLM-135M 모델을 로딩하고 평가합니다.
여기서 tokenizer는 텍스트(문장)를 모델이 이해할 수 있는 작은 단위(token)로 나누는 역할을 수행합니다.

In [5]:
# 3
model_path = "HuggingFaceTB/SmolLM-135M"
os.environ["HF_HOME"] = "D:\\data\\hf_cache"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, use_safetensors=True)

# Evaluate the model
model_perplexity = evaluate(model, tokenizer)
print(f"\nmodel perplexity: {model_perplexity:.2f}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

evaluating...: 100%|███████████████████████████████████████████████████████████████████████| 40/40 [00:14<00:00,  2.81it/s]


model perplexity: 16.54


## [실습 1] Magnitude-based Pruning 구현

Magnitude-based pruning 함수를 구현하세요. 구현은 CNN에서 magnitude-based pruning을 구현하는 방식과 유사합니다.

In [6]:
# 4
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            ##################### YOUR CODE STARTS HERE #####################
            num_elements = W.numel()
            num_zeros = round(num_elements * sparsity)
            importance = torch.abs(W)
            threshold = torch.kthvalue(importance.flatten(), num_zeros)[0]
            mask = importance > threshold
            ##################### YOUR CODE ENDS HERE #######################
            W.mul_(mask)

### 📘 셀 4 해설
**한 줄 요약:** [실습 1 정답] LLM의 각 Linear 층에서 **절댓값이 작은 가중치부터** `sparsity` 비율만큼 0으로 만드는 함수입니다(Magnitude-based Pruning).

- `@torch.no_grad()` : 함수 위에 붙는 **데코레이터**. 이 함수 안에서는 기울기 계산을 끄고 가중치를 직접 수정할 수 있게 합니다.
- `for n, m in model.named_modules():` : 모델 안의 모든 층을 `(이름, 층)`으로 하나씩 꺼냄
- `if isinstance(m, nn.Linear) and "lm_head" not in n:` : **Linear 층이면서** 이름에 `lm_head`가 **들어 있지 않은** 층만 대상으로 합니다.
  - `lm_head`는 마지막에 "다음 단어"의 점수를 내는 출력층입니다. 결과에 직접 영향을 주는 민감한 층이라 프루닝에서 제외합니다.
- `W = m.weight.data` : 그 층의 가중치 행렬 (모양: 출력 크기 × 입력 크기)

**정답 5줄**

| 코드 | 의미 |
|---|---|
| `num_elements = W.numel()` | 가중치 개수 |
| `num_zeros = round(num_elements * sparsity)` | 0으로 만들 개수 (반올림) |
| `importance = torch.abs(W)` | 중요도 = 절댓값 |
| `threshold = torch.kthvalue(importance.flatten(), num_zeros)[0]` | 중요도를 한 줄로 펴고 `num_zeros`번째로 **작은** 값을 찾음 = 커트라인. `kthvalue`는 (값, 위치)를 돌려주므로 `[0]`으로 값만 꺼냅니다. |
| `mask = importance > threshold` | 커트라인보다 크면 `True`(유지), 아니면 `False`(제거) |

- `W.mul_(mask)` : 마스크를 곱해 지울 자리만 0으로 만듭니다(`True`=1, `False`=0). 이름 끝의 `_`는 **원본을 직접 수정**한다는 표시입니다.

커트라인을 **층마다 따로** 구하므로, 모든 층이 똑같이 `sparsity` 비율만큼 지워집니다. 1번 실습(CNN)의 fine-grained pruning과 같은 방법입니다.

Pruning된 모델을 평가합니다.

In [7]:
# 5
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, use_safetensors=True)
prune_magnitude(model, 0.5)

# Evaluate the model
model_perplexity = evaluate(model, tokenizer)
print(f"\nmodel perplexity (magnitude 50%): {model_perplexity:.2f}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
evaluating...: 100%|███████████████████████████████████████████████████████████████████████| 40/40 [04:10<00:00,  6.27s/it]


model perplexity (magnitude 50%): 326.39


### 📘 셀 5 해설
**한 줄 요약:** 모델을 새로 불러와 **50%를 프루닝**한 뒤 perplexity를 측정합니다.

- `gc.collect()` : 파이썬에서 더 이상 쓰지 않는 메모리를 정리
- `torch.cuda.empty_cache()` : GPU에 잡혀 있던 여유 메모리를 반환. LLM은 메모리를 많이 쓰므로, 모델을 다시 불러오기 전에 **이전 모델의 흔적을 치우는** 단계입니다.
- `AutoModelForCausalLM.from_pretrained(model_path, ...)` : **원본 모델을 처음부터 다시 불러옵니다.** 프루닝은 가중치를 직접 바꿔 버리므로, 실험마다 깨끗한 모델로 시작해야 합니다.
  - `device_map="auto"` : GPU/CPU 배치를 자동으로 결정 / `torch_dtype=torch.float16` : 가중치를 16비트 실수로 불러옴
- `prune_magnitude(model, 0.5)` : 가중치의 절반을 제거
- `evaluate(model, tokenizer)` : perplexity 계산

**출력 해석:** perplexity `16.54 → 326.39`
- Perplexity는 낮을수록 좋은 지표인데 **약 20배** 나빠졌습니다. 다음 단어 후보가 16~17개 수준에서 300개가 넘는 수준으로 늘어난 셈이므로, 사실상 문장을 제대로 예측하지 못하는 상태입니다.
- 1번 실습의 CNN은 50%를 지워도 정확도가 거의 유지되었지만, **LLM은 단순한 크기 기준 프루닝에 훨씬 민감하다**는 것을 보여 줍니다. 그래서 더 정교한 기준(Wanda)이 필요합니다.

> `Some parameters are on the meta device because they were offloaded to the cpu.` 는 GPU 메모리가 부족해 모델 일부가 CPU로 옮겨졌다는 안내입니다. 이 때문에 평가가 4분가량 걸렸습니다(셀 3에서는 14초).

## [실습 2] Calibration 데이터셋 준비

[Wanda](https://arxiv.org/pdf/2306.11695)를 사용하여 importance를 계산하기 위해서는 calibration 데이터셋을 통해 activation을 추출해야 합니다.

이를 위해 ```get_calib_dataset``` 함수는 calibration dataset을 준비합니다.

In [ ]:
# 6
def get_calib_dataset(tokenizer=None, n_samples=256, block_size=512):
    dataset = load_dataset("mit-han-lab/pile-val-backup", split="validation")
    dataset = dataset.shuffle(seed=42)
    samples = []
    n_run = 0
    for data in dataset:
        line = data["text"]
        line = line.strip()
        line_encoded = tokenizer.encode(line)
        if len(line_encoded) > block_size:
            continue
        sample = torch.tensor([line_encoded])
        if sample.numel() == 0:
            continue
        samples.append(sample)
        n_run += 1
        if n_run == n_samples:
            break

    # now concatenate all samples and split according to block size
    cat_samples = torch.cat(samples, dim=1)
    n_split = cat_samples.shape[1] // block_size
    print(f" * Split into {n_split} blocks")
    return [cat_samples[:, i*block_size:(i+1)*block_size] for i in range(n_split)]

samples = get_calib_dataset(tokenizer)

### 📘 셀 6 해설
**한 줄 요약:** Wanda에 필요한 **calibration(보정)용 텍스트 데이터**를 준비합니다. 모델에 흘려보내 activation을 관찰하기 위한 샘플입니다.

**`get_calib_dataset(tokenizer=None, n_samples=256, block_size=512)`**
- `load_dataset("mit-han-lab/pile-val-backup", split="validation")` : 다양한 글이 모인 텍스트 데이터셋을 내려받음
- `dataset.shuffle(seed=42)` : 순서를 섞되, seed를 고정해 **항상 같은 순서**가 되게 함
- `for data in dataset:` 글을 하나씩 꺼내면서
  - `line = data["text"]`, `line.strip()` : 글 내용을 꺼내 앞뒤 공백 제거
  - `line_encoded = tokenizer.encode(line)` : 글을 **토큰 번호 리스트**로 변환
  - `if len(line_encoded) > block_size: continue` : 512토큰보다 긴 글은 **건너뜀**. `continue`는 "이번 차례는 넘기고 다음으로"라는 뜻입니다.
  - `sample = torch.tensor([line_encoded])` : 텐서로 변환 (모양: 1 × 토큰 수)
  - `samples.append(sample)`, `n_run += 1` : 리스트에 추가하고 개수 세기
  - `if n_run == n_samples: break` : 256개를 모으면 반복문을 **빠져나옴**
- `cat_samples = torch.cat(samples, dim=1)` : 모은 글들을 **한 줄로 길게 이어 붙임**
- `n_split = cat_samples.shape[1] // block_size` : 512토큰짜리 블록이 몇 개 나오는지 계산. `//`는 **몫만 구하는 나눗셈**입니다.
- `return [cat_samples[:, i*block_size:(i+1)*block_size] for i in range(n_split)]` : 512토큰씩 잘라 낸 블록들의 리스트. `[... for i in range(n)]`은 반복문으로 리스트를 만드는 **리스트 컴프리헨션**이고, `a:b`는 a번째부터 b번째 직전까지를 자르는 **슬라이싱**입니다.

마지막 줄 `samples = get_calib_dataset(tokenizer)`로 실제 데이터를 만들어 둡니다.

> 이 셀은 실행 결과가 저장되어 있지 않습니다. 실행하면 ` * Split into N blocks` 형태로 블록 개수가 출력됩니다.

아래 빈칸을 작성하여 `activation_norm`을 계산하세요.
Wanda 기법에서는 `activation_norm`을 L2 norm으로 계산하지만, 구현에서는 calibration 데이터셋에 대해 여러 번 반복하여 누적하므로
`activation_norm` 계산 시 제곱을 적용한 형태로 누적합니다. (이후 값을 사용할 때는 제곱근을 적용해야 합니다.)

In [ ]:
# 7
@torch.no_grad()
def get_calib_feat(model, tokenizer, samples):
    input_dict = dict()
    nsamples_dict = dict()
    def add_batch(m, x, y, name):
        if name not in input_dict:
            input_dict[name] = torch.zeros((m.weight.data.shape[1]), device=m.weight.data.device)
            nsamples_dict[name] = 0

        if isinstance(x, tuple):
            x = x[0]

        if len(x.shape) == 2:
            x = x.unsqueeze(0)
        tmp = x.shape[0]
        if len(x.shape) == 3:
            x = x.reshape((-1, x.shape[-1]))
        x = x.t()

        input_dict[name] *= nsamples_dict[name] / (nsamples_dict[name] + tmp)
        nsamples_dict[name] += tmp

        x = x.type(torch.float32)
        ##################### YOUR CODE STARTS HERE #####################
        # activation_norm을 계산하세요.
        # x.shape => (hidden_size, batch_size)
        activation_norm = torch.norm(x, p=2, dim=1) ** 2
        # activation_norm.shape => (hidden_size)
        ##################### YOUR CODE ENDS HERE #######################
        input_dict[name] += activation_norm / nsamples_dict[name]

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in name:
            hooks.append(
                m.register_forward_hook(
                    partial(add_batch, name=name)))

    print("Collecting norm of input activations...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    pbar = tqdm.tqdm(samples)
    for input_ids in pbar:
        input_ids = input_ids.to(device)
        model(input_ids)

    for key in input_dict.keys():
        input_dict[key].sqrt_()

    for hook in hooks:
        hook.remove()
    return input_dict

### 📘 셀 7 해설
**한 줄 요약:** [실습 2 정답] calibration 데이터를 모델에 통과시키면서, **각 Linear 층에 들어오는 입력(activation)의 크기**를 입력 채널별로 측정합니다.

**왜 필요할까?** 가중치가 작더라도, 거기에 곱해지는 **입력값이 항상 크다면** 결과에 미치는 영향은 큽니다. Wanda는 이 "입력의 크기"를 중요도 계산에 함께 사용합니다.

**hook(훅)** : 층에 "계산이 끝날 때마다 이 함수를 불러 줘"라고 걸어 두는 장치입니다. 모델 코드를 고치지 않고 중간 값을 볼 수 있습니다.

**`add_batch(m, x, y, name)` — 훅이 불릴 때마다 실행되는 함수** (`m`=층, `x`=층의 입력, `y`=층의 출력)

| 코드 | 의미 |
|---|---|
| `if name not in input_dict:` | 이 층을 처음 만났다면 |
| `input_dict[name] = torch.zeros((m.weight.data.shape[1]), ...)` | 입력 채널 수만큼 0으로 채운 기록 칸을 만들고 |
| `nsamples_dict[name] = 0` | 지금까지 본 샘플 수를 0으로 |
| `if isinstance(x, tuple): x = x[0]` | 훅은 입력을 튜플로 넘겨주므로 실제 텐서만 꺼냄 |
| `if len(x.shape) == 2: x = x.unsqueeze(0)` | 2차원이면 맨 앞에 차원을 하나 추가해 3차원으로 통일 |
| `tmp = x.shape[0]` | 이번에 들어온 샘플 수 |
| `x = x.reshape((-1, x.shape[-1]))` | (샘플, 토큰, 채널) → (전체 토큰 수, 채널) |
| `x = x.t()` | 행·열을 바꿈 → (채널, 전체 토큰 수) |
| `input_dict[name] *= nsamples_dict[name] / (nsamples_dict[name] + tmp)` | 기존 기록의 비중을 줄임 (**누적 평균**을 위한 준비) |
| `nsamples_dict[name] += tmp` | 샘플 수 갱신 |
| `x = x.type(torch.float32)` | 계산 정밀도를 위해 32비트 실수로 변환 |
| **(정답)** `activation_norm = torch.norm(x, p=2, dim=1) ** 2` | 채널별 L2 norm의 **제곱** |
| `input_dict[name] += activation_norm / nsamples_dict[name]` | 평균에 반영 |

- **L2 norm** : 값들을 제곱해서 더한 뒤 제곱근을 취한 것. `dim=1`은 토큰 방향으로 계산하라는 뜻이므로, **채널마다 숫자 1개**가 나옵니다.
- `** 2`로 제곱해 두는 이유: 여러 묶음의 결과를 **평균 내려면 제곱 상태로 더해야** 맞기 때문입니다. 마지막에 한 번만 제곱근을 취합니다.

**함수 본문**
- `partial(add_batch, name=name)` : `add_batch`의 `name` 인자를 **미리 채워 넣은** 새 함수를 만듭니다. 층마다 자기 이름으로 기록되게 하기 위해서입니다.
- `m.register_forward_hook(...)` : `lm_head`를 제외한 모든 Linear 층에 훅을 겁니다.
- `for input_ids in pbar: model(input_ids)` : 블록을 하나씩 모델에 통과시킴 → 훅들이 값을 기록 (`tqdm`은 진행률 막대)
- `input_dict[key].sqrt_()` : 누적해 둔 제곱 값에 **제곱근**을 취해 최종 norm으로 만듦
- `hook.remove()` : 기록이 끝났으니 훅 제거
- `return input_dict` : `{층 이름: 입력 채널별 activation 크기}` 딕셔너리

> 코드 주석의 `x.shape => (hidden_size, batch_size)`에서 두 번째 값은 정확히는 "그 묶음의 전체 토큰 수"입니다.

In [ ]:
# 8
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, use_safetensors=True)
input_feat = get_calib_feat(model, tokenizer, samples)

### 📘 셀 8 해설
**한 줄 요약:** 깨끗한 모델을 다시 불러와, 방금 만든 함수로 **activation 크기를 실제로 수집**합니다.

- `gc.collect()`, `torch.cuda.empty_cache()` : 메모리 정리
- `model = AutoModelForCausalLM.from_pretrained(...)` : 셀 5에서 프루닝된 모델 대신 **원본 모델**을 다시 불러옴 (activation은 프루닝 전 모델 기준으로 재야 하므로)
- `input_feat = get_calib_feat(model, tokenizer, samples)` : 결과를 `input_feat`에 저장

`input_feat`은 딕셔너리이고, 예를 들어 `input_feat['어떤 층 이름']`은 그 층의 **입력 채널 수만큼의 숫자**를 담은 텐서입니다. 값이 클수록 "그 채널로 들어오는 입력이 평소에 크다"는 뜻입니다.

> 이 셀은 실행 결과가 저장되어 있지 않습니다. 실행하면 `Collecting norm of input activations...`와 진행률 막대가 출력됩니다.

## [실습 3] Wanda Pruning 구현

Wanda 논문을 참고하여 아래 빈 칸을 채워 Wanda Pruning을 구현하고 실행해보세요.

Wanda에서는 일반적인 magnitude-based pruning 방법과 다르게게 weight parameter의 행(row)별로 동일한 희소성(sparsity)으로 pruning을 진행합니다.

In [ ]:
# 9
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            ##################### YOUR CODE STARTS HERE #####################
            row, col = W.shape
            num_zeros_per_row = round(col * sparsity)
            importance = torch.abs(W) * input_feat[n]
            threshold = torch.kthvalue(importance, num_zeros_per_row, dim=1)[0]
            mask = importance > threshold.reshape(row, 1)
            ##################### YOUR CODE ENDS HERE #######################
            W.mul_(mask)

### 📘 셀 9 해설
**한 줄 요약:** [실습 3 정답] **Wanda** 방식의 프루닝 함수입니다. 중요도를 `|가중치| × 입력 activation의 크기`로 계산하고, **행(row)마다** 같은 비율로 지웁니다.

셀 4(Magnitude)와 틀은 같고, 정답 부분만 다릅니다.

| 코드 | 의미 |
|---|---|
| `row, col = W.shape` | 가중치 행렬의 (행 수, 열 수) = (출력 크기, 입력 크기) |
| `num_zeros_per_row = round(col * sparsity)` | **한 행에서** 0으로 만들 개수 |
| `importance = torch.abs(W) * input_feat[n]` | 중요도 = 절댓값 × 그 열(입력 채널)의 activation 크기 |
| `threshold = torch.kthvalue(importance, num_zeros_per_row, dim=1)[0]` | **행마다** k번째로 작은 값 = 행별 커트라인 (행 수만큼 나옴) |
| `mask = importance > threshold.reshape(row, 1)` | 각 행을 자기 커트라인과 비교 |

- `torch.abs(W) * input_feat[n]` : `W`는 (행, 열) 모양이고 `input_feat[n]`은 (열) 모양입니다. 모양이 달라도 PyTorch가 자동으로 맞춰 계산해(**브로드캐스팅**) **j번째 열 전체에 j번째 activation 크기**가 곱해집니다.
- `dim=1` : 열 방향으로, 즉 **각 행 안에서** k번째 값을 찾으라는 뜻
- `threshold.reshape(row, 1)` : (행) 모양을 (행, 1)로 바꿔야 각 행이 자기 커트라인과 비교됩니다.

**Magnitude와 Wanda 비교**

| | Magnitude (셀 4) | Wanda (셀 9) |
|---|---|---|
| 중요도 | \|W\| | \|W\| × 입력 activation 크기 |
| 비교 범위 | 층 전체에서 한 번 | **행마다 따로** |
| 필요한 것 | 가중치만 | 가중치 + calibration 데이터 |

**작은 예시:** 가중치 0.1에 평소 입력이 50인 채널(중요도 5)과, 가중치 1.0에 평소 입력이 0.5인 채널(중요도 0.5)이 있다면, Magnitude는 앞의 것을 지우지만 Wanda는 **뒤의 것**을 지웁니다. 실제 출력에 미치는 영향은 앞의 것이 더 크기 때문입니다.

행마다 따로 비교하면, 모든 출력 뉴런이 **같은 개수의 연결**을 유지하게 되어 특정 뉴런의 연결이 몽땅 사라지는 일을 막을 수 있습니다.

In [ ]:
# 10
gc.collect()
torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.float16, use_safetensors=True)
prune_wanda(model, sparsity=0.5, input_feat=input_feat)

# Evaluate the model
model_perplexity = evaluate(model, tokenizer)
print(f"\nmodel perplexity (wanda 50%): {model_perplexity:.2f}")

### 📘 셀 10 해설
**한 줄 요약:** 모델을 다시 불러와 **Wanda로 50%를 프루닝**하고 perplexity를 측정합니다. 구성은 셀 5와 같습니다.

- `prune_wanda(model, sparsity=0.5, input_feat=input_feat)` : 셀 8에서 수집한 activation 크기를 함께 넘겨줍니다. `이름=값` 형태로 인자를 넘기는 **키워드 인자** 방식입니다.

**결과 확인 방법:** 이 셀은 실행 결과가 저장되어 있지 않아 **수치를 확인할 수 없습니다.** 실행하면 `model perplexity (wanda 50%): ...`가 출력되며, 아래 두 값과 비교하면 됩니다.

| 모델 | Perplexity |
|---|---|
| 원본 (셀 3) | 16.54 |
| Magnitude 50% (셀 5) | 326.39 |
| Wanda 50% (셀 10) | (실행 후 확인) |

같은 50%를 지우더라도 Wanda의 값이 Magnitude보다 원본에 얼마나 더 가까운지가 이 실습의 관찰 포인트입니다.

**전체 실습 정리**
1. LLM은 단순히 절댓값이 작은 가중치를 지우는 것만으로는 성능이 크게 무너집니다.
2. Wanda는 **가중치의 크기와 입력의 크기를 함께** 고려해 "실제로 출력에 영향이 작은" 가중치를 골라 지웁니다.
3. 이를 위해 소량의 calibration 데이터를 한 번 통과시키기만 하면 되고, **재학습은 필요하지 않습니다.**